# 🧠 NeuroScope AI — Meningioma (BraTS-MEN) 3D Transfer Learning on GPU
**Project:** BrainAgenticthon 2026 — Medical AI Workstream (Rou'a & Abdulrahman)  
**Objective:** Adapt our trained 3D SegResNet model from Glioma to **Meningioma (الأورام السحائية)** using Transfer Learning.

---
### ⚡ Why Transfer Learning from Glioma to Meningioma?
- **Pretrained Anatomical Features:** The Glioma model already understands 3D brain MRI geometry, skull contours, and contrast enhancement.
- **Fast Convergence:** Needs only **5 to 10 epochs** on GPU (finishes in **~5 to 8 minutes** on Tesla T4!).
- **Gold Standard Dataset:** Official BraTS 2023 Meningioma (`MedOtter/brats2023-men-dataset`) streamed directly from Hugging Face.

## Step 1: Verify GPU Acceleration (Tesla T4 / A100)
Ensure `Runtime` ➔ `Change runtime type` is set to **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Please switch to GPU runtime via Runtime -> Change runtime type!")

## Step 2: Clone Repository & Install MONAI

In [ ]:
import os
if not os.path.exists("MRI-"):
    !git clone https://github.com/M0maniiR0/MRI-.git
%cd /content/MRI-
!pip install -q monai nibabel scipy

## Step 3: Stream BraTS Meningioma Cohort from Hugging Face (~1 Minute)
Streams 25 official 3D Meningioma cases (T1, T1ce, T2, FLAIR, Mask) directly from Hugging Face.

In [ ]:
%cd /content/MRI-
!python download_meningioma_data.py --dest Task_Meningioma/BraTS_MEN --num-cases 25

## Step 4: Launch Transfer Learning on GPU (5-8 Minutes Total)
Adapts the pretrained Glioma SegResNet weights to Meningioma with `DiceCELoss` and high-resolution $96^3$ crops.

In [ ]:
%cd /content/MRI-
!python train_meningioma_transfer.py \
    --data Task_Meningioma/BraTS_MEN \
    --pretrained models/segresnet_fullvolume_best.pt \
    --output models/segresnet_meningioma_best.pt \
    --epochs 10 \
    --lr 1e-4 \
    --device cuda

## Step 5: Save Checkpoint to Google Drive & Download

In [ ]:
%cd /content/MRI-
from google.colab import files
import os
if os.path.exists("models/segresnet_meningioma_best.pt"):
    files.download("models/segresnet_meningioma_best.pt")
    print("Downloaded models/segresnet_meningioma_best.pt successfully!")
else:
    print("Checkpoint not found. Run training first.")